# Bloque 1: Instalación y Carga de Datos

In [13]:
# Instala/actualiza la librería de Machine Learning
# pip install -U scikit-learn

# Librería para manejar tablas de datos (DataFrames)
import pandas as pd

# Carga el archivo CSV
df = pd.read_csv('df_total.csv', encoding='UTF-8')
df.head() # Muestra las primeras filas

# Vemos un ejemplo de noticia sin procesar
print(f"Noticia original: {df['news'][3]}")

Noticia original: Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual respecto al avance del 30 de marzo y se sitúa 22 puntos por encima del dato de febrero que ascendió al 76.De esos 22 puntos de diferencia la mayor parte la colocó el grupo de la vivienda 09 puntos por la subida de la electricidad y el del transporte 07 puntos por el alza de los carburantes. También impulsaron el IPC de marzo el aumento de los precios de la restauración y los servicios de alojamiento y al encarecimiento generalizado de los alimentos especialmente del pescado y el marisco de la carne de las legumbres y hortalizas y de la leche el queso y los huevos.Sin tener en cuenta la rebaja del impuesto especial sobre la electricidad y las variaciones sobre otros impuestos el IPC interanual alcanzó en marzo 107 nueve décimas más que la tasa general del 98. Así lo refleja el IPC a impuestos constantes que el INE también pu

In [14]:
df.columns

Index(['url', 'news', 'Type'], dtype='object')

# Bloque 2: Preparación y División de Datos

In [15]:
from sklearn.model_selection import train_test_split

X = df['news'] # Variable independiente (el texto de la noticia)
y = df['Type'] # Variable dependiente (la etiqueta o categoría)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bloque 3: Vectorización (Convertir texto a números)

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

#Vemos
X_train_transformed #te mantiene las filas de datos, pero tiene unvector de n columnas
X_train_transformed_dense = X_train_transformed.toarray()

print(X_train_transformed_dense)

[[0 2 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 2 0 ... 0 0 0]]


# Bloque 4: Creación del Modelo y

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

model = MultinomialNB() # Algoritmo basado en probabilidad, excelente para texto
model.fit(X_train_transformed, y_train) # ¡Aquí ocurre el aprendizaje!
y_pred = model.predict(X_test_transformed) # El modelo intenta adivinar las etiquetas del test
print(metrics.accuracy_score(y_test, y_pred)) # Comparamos aciertos vsrealidad

0.7991803278688525


# Bloque 5: Mejora con Stemming (Raíces de palabras)

In [18]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

nltk.download('punkt')
stemmer = SnowballStemmer('spanish')

# Función para procesar el texto
def tokenize_and_stem(text):
    tokens = word_tokenize(text.lower())

    # Solo nos quedamos con letras (quitamos signos de puntuación) y aplicamos stemmer
    stems = [stemmer.stem(token) for token in tokens if

token.isalpha()]
    
    return ' '.join(stems)
    
# Aplicamos la función a todo el dataset
df['news_stemmer'] = df['news'].apply(tokenize_and_stem)

# REPETIMOS EL PROCESO CON LOS NUEVOS DATOS
X = df['news_stemmer']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# IMPORTANTE: Hay que volver a ajustar el vectorizador a las nuevas palabras (raíces)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(f"Precisión con Stemming: {metrics.accuracy_score(y_test, y_pred)}")

[nltk_data] Downloading package punkt to /home/ciabd14/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Precisión con Stemming: 0.8278688524590164


# Bloque 6: Mejora con Lematización (Diccionario)

In [19]:
import spacy

# Cargamos el modelo en español de spacy
nlp = spacy.load('es_core_news_sm')

def lemmatize_text(text):
    doc = nlp(text.lower())

    # Extraemos el lema de cada palabra si es una letra
    lemmas = [token.lemma_ for token in doc if token.is_alpha]
    return ' '.join(lemmas)

# Aplicamos lematización (esto puede tardar unos minutos)
df['news_lemma'] = df['news'].apply(lemmatize_text)

# REPETIMOS EL PROCESO CON LOS DATOS LEMATIZADOS
X = df['news_lemma']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Volvemos a transformar los datos
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)
print(f"Precisión con Lematización: {metrics.accuracy_score(y_test, y_pred)}")

Precisión con Lematización: 0.8319672131147541
